# Divisão espacial k-fold dos patches (estágio 07)

Atribui cada patch registrado no manifesto do estágio 06 a uma das `splits.fold_count` dobras (k=5) por proximidade geográfica: os centroides das bboxes são agrupados por k-means determinístico (implementado em numpy puro, semente fixa) e os grupos são rotulados deterministicamente, preservando a contiguidade espacial para reduzir o vazamento por autocorrelação entre treino e validação nos estágios 09/10. A divisão é gravada na coluna `fold` do próprio `MyDrive/tcc/data/processed/manifest.parquet` e versionada em `split.meta.json` (fingerprint do manifesto + dobras + semente), de modo que execuções repetidas reutilizam a divisão vigente sem reprocessamento. O manifesto é reutilizado do estágio 06 (idempotência) e a figura de registro vai para `MyDrive/tcc/artifacts/figures/`.

## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.

In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/jotap1101/tcc/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")

## Dependências pinadas

Instala as versões fixadas em `requirements-runtime.txt`, garantindo o mesmo conjunto de bibliotecas nas duas plataformas.

In [ ]:
# Instala as versões pinadas do requirements-runtime.txt no ambiente atual.
import subprocess
import sys

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)
print("Dependências instaladas a partir de:", requirements)

## Atualização dos módulos `src/` em memória

Remove do cache do kernel (`sys.modules`) os módulos `src.*` carregados em execuções anteriores, garantindo que as próximas importações usem a versão recém-sincronizada pelo bootstrap (evita módulos obsoletos após edições do código).

In [ ]:
# Remove os módulos src.* em cache para forçar o carregamento da versão atual do workspace.
import sys

for module in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[module]
print("Módulos src.* recarregados do workspace sincronizado.")

## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve todos os caminhos de armazenamento definidos em `src/config.yaml`, incluindo a subpasta do manifesto e das figuras.

In [ ]:
# Importa o pacote compartilhado, identifica a plataforma e resolve os caminhos de armazenamento.
from src import io
from src.config import get_config

platform = io.detect_platform()
storage_paths = io.resolve_storage_paths()
config = get_config()
print(f"Plataforma: {platform}")
print(f"Manifesto: {storage_paths['data_processed'] / 'manifest.parquet'}")
print(f"Figuras: {storage_paths['artifacts_figures']}")

## Reprodutibilidade

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch, garantindo o mesmo protocolo de execução nas duas plataformas.

In [ ]:
# Fixa sementes e flags determinísticas do PyTorch de acordo com a configuração.
from src.utils import set_all_seeds, set_deterministic_flags

set_all_seeds(config["reproducibility"]["seed"])
set_deterministic_flags()
print(f"Seed fixada: {config['reproducibility']['seed']}")

## Dependência do estágio 06

Verifica que o manifesto de patches do estágio 06 está disponível no caminho canônico e informa se ele está vigente frente às entradas atuais (composite e máscara final) — sem o manifesto a divisão espacial não pode prosseguir.

In [ ]:
# Verifica a dependência do estágio 06 (manifesto de patches).
from src.data.patch_generation import manifest_is_current, manifest_path

manifest_file = manifest_path(storage_paths)
if not io.path_exists(manifest_file):
    raise FileNotFoundError(f"Manifesto do estágio 06 não encontrado: {manifest_file}")
print(f"Disponível: Manifesto (estágio 06): {manifest_file}")
print(f"Manifesto vigente frente às entradas: {manifest_is_current(storage_paths)}")

## Divisão espacial k-fold

Garante (idempotente) a divisão espacial k-fold: os patches do manifesto do estágio 06 são agrupados por proximidade geográfica dos centroides (k-means determinístico em numpy puro, com semente fixa) e cada grupo recebe uma dobra de 0 a k-1, com rótulos ordenados deterministicamente pela posição dos centros. A coluna `fold` é gravada no manifesto e o metadata da divisão (`split.meta.json`) registra o fingerprint — divisão vigente é reutilizada em reexecuções.

In [ ]:
# Garante a divisão espacial k-fold (reutiliza se já existir).
from src.data.spatial_split import assign_spatial_folds

split_manifest = assign_spatial_folds(storage_paths)

## Verificação da divisão

Confere a divisão persistida: total de patches e tiles, contagem de patches, proporção média de café e localização média (centroide) de cada dobra.

In [ ]:
# Verifica a divisão persistida (contagens e localização por dobra).
from src.data.spatial_split import verify_split

split_stats = verify_split(storage_paths)
print(f"Dobras: {split_stats['fold_count']} | Patches: {split_stats['n_patches']} | Tiles: {split_stats['n_tiles']}")
for row in split_stats["per_fold"]:
    print(
        f"  dobra {row['fold']}: {row['n_patches']} patches | café {row['coffee_ratio_mean']:.3f} | "
        f"centroide ({row['centroid_x']:.0f}, {row['centroid_y']:.0f})"
    )

## Figura da divisão espacial

Renderiza e persiste o mapa de centroides dos patches coloridos por dobra em `MyDrive/tcc/artifacts/figures/`; execuções repetidas reutilizam a figura já existente (idempotência).

In [ ]:
# Renderiza e persiste a figura da divisão espacial (reutiliza se já existir).
from src.data.spatial_split import save_split_figure

split_figure = save_split_figure(storage_paths)

## Resumo da etapa

Exibe o resumo da divisão espacial k-fold: manifesto reutilizado, divisão persistida, distribuição por dobra e figura de registro.

In [ ]:
# Exibe o resumo da etapa de divisão espacial k-fold.
summary = {
    "Manifesto de entrada (estágio 06)": str(manifest_file),
    "Divisão espacial k-fold": str(split_manifest),
    "Dobras": split_stats["fold_count"],
    "Patches divididos": split_stats["n_patches"],
    "Tiles envolvidos": split_stats["n_tiles"],
    "Figura da divisão": str(split_figure),
}
for key, value in summary.items():
    print(f"{key}: {value}")
print("Estágio 07 concluído.")